# Step 4: RAG (Retrieval-Augmented Generation) with LangChain

This notebook covers the RAG tutorial from the official LangChain docs:
https://python.langchain.com/docs/tutorials/rag/

**Important note before you start:** the live tutorial page has recently been rewritten to use a
newer `create_agent` + tool-calling / middleware pattern (and it now loads pages with
`requests` + `BeautifulSoup` instead of `WebBaseLoader`, and skips `create_retrieval_chain` /
`create_stuff_documents_chain` entirely). Those older two functions still exist in LangChain and
are **not deprecated legacy chains** (they are not the old `RetrievalQA` style) — they're the
LCEL-based "two-step RAG chain" pattern that most courses (including yours) still teach at this
stage, so this notebook builds the pipeline that way, exactly as you asked. Everywhere else,
we use the current, non-deprecated LangChain APIs.

**Your setup for this notebook:**
- LLM: `ChatGroq` with `llama-3.3-70b-versatile` (instead of OpenAI)
- Embeddings: `HuggingFaceEmbeddings` with `all-MiniLM-L6-v2` (free, local, no API key)
- Vector store: `Chroma`
- API key loaded from a `.env` file with `python-dotenv`
- No LangSmith tracing, no OpenAI anything, no deprecated chains, no streaming

**Two packaging quirks you'll see and can ignore:**
1. You'll get a `DeprecationWarning` when importing from `langchain_community` — that package
   is being sunset in favor of standalone integration packages (e.g. `langchain-huggingface`
   instead of `langchain_community.embeddings`). It still works fine; this notebook uses it
   because you specifically asked to import `HuggingFaceEmbeddings` from there.
2. As of **LangChain v1.0**, `create_stuff_documents_chain` and `create_retrieval_chain` were
   moved out of the core `langchain` package into a new `langchain-classic` package. The
   install and import cells below handle this automatically either way.


## 0. Install & import dependencies

Run this once. It installs everything needed for this notebook:
- `langchain` core pieces + `langchain-groq` for the LLM
- `langchain-community` for `WebBaseLoader`
- `langchain-huggingface`'s backing libraries (`sentence-transformers`) for free local embeddings
- `langchain-chroma` for the vector store
- `beautifulsoup4` because `WebBaseLoader` uses it under the hood to parse HTML
- `python-dotenv` to load your `GROQ_API_KEY` from a `.env` file


In [ ]:
# Install dependencies (safe to re-run; pip will skip what's already installed)
#
# Note: as of LangChain v1.0, `create_stuff_documents_chain` and `create_retrieval_chain`
# were moved OUT of the core `langchain` package and into a separate `langchain-classic`
# package. Without installing langchain-classic, `from langchain.chains import ...` will
# raise ModuleNotFoundError even though `langchain` itself installed fine.
%pip install -q langchain langchain-classic langchain-groq langchain-community langchain-chroma \
    langchain-text-splitters sentence-transformers beautifulsoup4 python-dotenv


In [ ]:
# --- Core imports used throughout the notebook ---
import os
from dotenv import load_dotenv  # reads key=value pairs out of a .env file

# Document loading
from langchain_community.document_loaders import WebBaseLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings (free, local, no OpenAI key needed)
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma

# LLM (Groq instead of OpenAI)
from langchain_groq import ChatGroq

# Prompting + the two chain builders that assemble the full RAG pipeline
from langchain_core.prompts import ChatPromptTemplate

# As of LangChain v1.0+, these two functions live in the separate `langchain-classic`
# package instead of the core `langchain` package. This try/except makes the notebook
# work whether you have an older langchain (<1.0, where they're still in `langchain.chains`)
# or a current one (>=1.0, where they're in `langchain_classic.chains`).
try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains import create_retrieval_chain
except ModuleNotFoundError:
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.chains import create_retrieval_chain


In [ ]:
# Load environment variables from a local .env file.
# Your .env file should contain a line like:
#   GROQ_API_KEY=gsk_your_actual_key_here
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY not found. Make sure it's set in your .env file."
print("Groq API key loaded successfully.")

# WebBaseLoader sends HTTP requests to fetch pages; setting USER_AGENT identifies
# your requests and silences a harmless warning from langchain_community.
os.environ.setdefault("USER_AGENT", "langchain-rag-tutorial/1.0")


## 1. Document Loading — `WebBaseLoader`

**What this does, in simple terms:** before you can ask questions about a web page, LangChain
needs to actually go fetch that page and turn it into a `Document` object (basically: text +
some metadata like the source URL). `WebBaseLoader` does exactly that — it downloads the HTML
and strips it down to readable text for you, so you don't have to write your own `requests` +
`BeautifulSoup` code.

Here we load the same example page the official tutorial uses (Lilian Weng's blog post on LLM
agents), and we scope the parser to only the post's title, header, and content — this avoids
pulling in navbars, sidebars, and footer junk.


In [ ]:
import bs4  # BeautifulSoup, used here only to restrict which HTML tags WebBaseLoader keeps

# Only keep the elements that actually contain the article itself.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},  # ignore everything else on the page
)

docs = loader.load()  # returns a list of Document objects (one per URL here)

assert len(docs) == 1
print(f"Total characters loaded: {len(docs[0].page_content)}")
print(docs[0].page_content[:500])  # peek at the first 500 characters


## 2. Text Splitting — `RecursiveCharacterTextSplitter`

**What this does, in simple terms:** the whole blog post is way too long to hand to an LLM in
one go (and even if it fit, LLMs are worse at finding facts buried in a huge wall of text).
So we chop the document into smaller overlapping chunks. `RecursiveCharacterTextSplitter` tries
to split on natural boundaries first (paragraphs, then sentences, then words) so chunks don't
get cut off mid-sentence whenever possible.

- `chunk_size=1000`: each chunk is roughly 1000 characters.
- `chunk_overlap=200`: consecutive chunks share 200 characters, so context isn't lost at the
  edges of a chunk.
- `add_start_index=True`: keeps track of where each chunk started in the original document.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # target size of each chunk, in characters
    chunk_overlap=200,    # overlap between consecutive chunks, in characters
    add_start_index=True, # store each chunk's starting position in the original doc
)

all_splits = text_splitter.split_documents(docs)  # split every loaded Document

print(f"Split the blog post into {len(all_splits)} chunks.")
print(all_splits[0].page_content[:300])  # preview the first chunk


## 3. Embeddings — `HuggingFaceEmbeddings` (free, no OpenAI key needed)

**What this does, in simple terms:** an embedding model turns a chunk of text into a list of
numbers (a vector) that captures its *meaning*. Chunks with similar meaning end up with vectors
that are close together in that numeric space. This is what makes "search by meaning" possible,
instead of just keyword matching.

We use `all-MiniLM-L6-v2`, a small, fast, free sentence-transformer model that runs locally on
your machine — no API key, no cost, no internet round-trip per call (after the model is
downloaded once).


In [ ]:
# Free local embedding model - runs on your machine, no API key required
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Quick sanity check: embed a single string and look at the vector shape
sample_vector = embeddings.embed_query("What is task decomposition?")
print(f"Embedding vector length: {len(sample_vector)}")
print(sample_vector[:5], "...")  # print just the first 5 numbers


## 4. Vector Store — `Chroma`

**What this does, in simple terms:** now that we can turn text into vectors, we need somewhere
to *store* all those vectors alongside their original text chunks, so we can later search
through them efficiently. `Chroma` is a lightweight, open-source vector database that runs
locally — perfect for learning and small projects.

We create a `Chroma` store configured with our HuggingFace embedding function, then add all of
our chunks to it. Under the hood, each chunk gets embedded and stored together with its vector.


In [ ]:
# Create (or open) a local Chroma vector store on disk
vector_store = Chroma(
    collection_name="langchain_rag_tutorial",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # saved locally so it persists between runs
)

# Embed every chunk and add it to the vector store
document_ids = vector_store.add_documents(documents=all_splits)

print(f"Added {len(document_ids)} chunks to the vector store.")
print("First few IDs:", document_ids[:3])


## 5. Retriever — `vectorstore.as_retriever()`

**What this does, in simple terms:** a `VectorStore` alone knows how to do similarity search,
but to plug it into a LangChain *chain*, it's cleanest to wrap it as a `Retriever` — a standard
LangChain interface for "give me a query, get back relevant documents". `as_retriever()` does
exactly that conversion.

`search_kwargs={"k": 4}` tells it to return the top 4 most relevant chunks for any given query.


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",   # plain similarity search (closest vectors win)
    search_kwargs={"k": 4},     # return the top 4 most relevant chunks
)

# Quick test: what does the retriever pull back for a sample question?
retrieved_docs = retriever.invoke("What is task decomposition?")
print(f"Retrieved {len(retrieved_docs)} chunks.\n")
print(retrieved_docs[0].page_content[:300])


## 6. Full RAG Pipeline — `create_stuff_documents_chain` + `create_retrieval_chain`

**What this does, in simple terms:** this is where everything comes together into one pipeline
you can just call with a question and get an answer back.

- `create_stuff_documents_chain` builds a small chain that takes retrieved documents, "stuffs"
  their text into a prompt template alongside the user's question, and sends that to the LLM.
- `create_retrieval_chain` wraps that around the retriever, so the full flow becomes:
  **question → retriever fetches relevant chunks → stuff chain builds the prompt → LLM
  (ChatGroq) generates the answer.**

We use `ChatGroq` with `llama-3.3-70b-versatile` as the LLM instead of OpenAI.


In [ ]:
# LLM: Groq-hosted Llama 3.3 70B instead of OpenAI's GPT models
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0,  # keep answers focused and deterministic for Q&A
)

# Prompt template: {context} will be filled with retrieved chunks, {input} with the user's question
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Step A: chain that "stuffs" retrieved docs into the prompt and calls the LLM
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Step B: wraps the retriever + the above chain into one end-to-end RAG pipeline
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


In [ ]:
# Ask a question and run the full pipeline
response = rag_chain.invoke({"input": "What is task decomposition?"})

print("ANSWER:\n", response["answer"])
print("\n---\n")
print(f"Answer was generated from {len(response['context'])} retrieved chunks.")


In [ ]:
# Try a couple more questions to see the pipeline in action
for question in [
    "What are the main components of an autonomous LLM agent?",
    "What is Chain of Thought prompting?",
]:
    result = rag_chain.invoke({"input": question})
    print(f"Q: {question}")
    print(f"A: {result['answer']}\n")


## Comparison: LangChain Components vs. Your From-Scratch RAG Pipeline

If you previously built RAG manually (without LangChain), here's how each piece you wrote by
hand maps onto the LangChain components used above:

| Stage | Your from-scratch code | LangChain component (this notebook) | What it's doing |
|---|---|---|---|
| **Document loading** | Manually fetching a URL (e.g. `requests.get`) and stripping HTML/reading a file into a string | `WebBaseLoader` | Downloads the page and wraps the text in a standard `Document` object with metadata |
| **Chunking** | A custom loop slicing the string every N characters (maybe with manual overlap logic) | `RecursiveCharacterTextSplitter` | Splits text on natural boundaries (paragraphs → sentences → words) with configurable size/overlap |
| **Embedding** | Calling a model (e.g. `sentence-transformers`) yourself and storing raw vectors in a list/array | `HuggingFaceEmbeddings` | Same underlying model, but wrapped in a consistent `.embed_query()` / `.embed_documents()` interface that plugs into everything else |
| **Vector store** | A NumPy array (or dict) of vectors + a manual cosine-similarity function to search it | `Chroma` | A real vector database: stores vectors + text + metadata, and does the similarity search for you, persisted to disk |
| **Retrieval** | Your manual "loop over all vectors, compute similarity, sort, take top-k" function | `vectorstore.as_retriever()` | Wraps that same idea (`k` nearest neighbors) behind a standard `.invoke(query)` interface |
| **Generation** | Manually formatting a prompt string with the retrieved text + question, then calling an LLM API | `create_stuff_documents_chain` + `create_retrieval_chain` | Automates "stuff retrieved chunks into a prompt template" and chains it directly to the retriever + LLM call, in one `.invoke()` |

**The big picture:** LangChain isn't doing anything conceptually different from your from-scratch
version — it's the *same* five stages (load → split → embed → store → retrieve → generate). What
you get from LangChain is standard interfaces between the stages, so you can swap out any one
piece (e.g. Chroma for another vector store, or Groq for another LLM) without rewriting the rest
of the pipeline.


## Practice Exercise: Build RAG Over a Different URL

Your turn! Pick any article, blog post, or documentation page you're interested in, and rebuild
the same pipeline for it. Fill in the `YOUR_URL_HERE` placeholder below and, if the page uses
different CSS classes for its content, adjust the `bs4_strainer` (or just remove `bs_kwargs`
entirely to load the whole page).

Try to do it without looking back at the cells above — that's the best way to check it actually
stuck.


In [ ]:
# --- Practice: RAG over a URL of your choice ---

practice_url = "YOUR_URL_HERE"  # <-- replace with any article/blog/docs URL you like

# 1) Load
practice_loader = WebBaseLoader(web_paths=(practice_url,))
practice_docs = practice_loader.load()

# 2) Split
practice_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
practice_splits = practice_splitter.split_documents(practice_docs)

# 3) Embed + 4) Store (reusing the same free embeddings model from Step 3)
practice_vector_store = Chroma(
    collection_name="practice_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_practice_db",
)
practice_vector_store.add_documents(practice_splits)

# 5) Retriever
practice_retriever = practice_vector_store.as_retriever(search_kwargs={"k": 4})

# 6) Full pipeline (reusing the same llm and prompt from Step 6)
practice_qa_chain = create_stuff_documents_chain(llm, prompt)
practice_rag_chain = create_retrieval_chain(practice_retriever, practice_qa_chain)

# Try asking it something about the page you chose!
practice_question = "Summarize the main point of this page."
practice_response = practice_rag_chain.invoke({"input": practice_question})
print(practice_response["answer"])
